# 08 — Price Trajectory

Demonstrates day-by-day Monte Carlo price trajectory forecasting:

1. **Regime-weighted drift/vol** — matrix-power regime probabilities × empirical stats
2. **Shared MC paths** — single set of Student-t random walks sampled at each day
3. **Analytical vs MC expected price** — median when empirical vol is used
4. **P(up) via Student-t survival** — probability of being above current price
5. **Scenario probabilities** — Down >5%, Down 3-5%, Flat +/-3%, Up 3-5%, Up >5%

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.data.prices import fetch_historical_prices
from research.models.markov import (
    classify_regime_series,
    estimate_transition_matrix,
    compute_markov_forecast,
)
from research.models.trajectory import (
    RegimeStats,
    compute_trajectory,
    compute_horizon_drift_vol,
    compute_scenario_probabilities,
)

## 1. Load Prices

Fetch historical closes and compute returns.

In [ ]:
TICKER = 'BTC'
DAYS = 180
HORIZON = 7

prices = fetch_historical_prices(TICKER, days=DAYS)
prices['returns'] = prices['close'].pct_change()
returns = prices['returns'].dropna().values
current_price = float(prices['close'].iloc[-1])

print(f"Current price: ${current_price:,.2f}")
print(f"Returns: {len(returns)} days")

## 2. Regime Classification & Transition Matrix

Estimate the transition matrix and empirical regime statistics.

In [ ]:
regimes = classify_regime_series(returns)
P = estimate_transition_matrix(regimes, decay_rate=0.97)
current_regime = regimes[-1]

# Empirical regime statistics
regime_stats = {}
for state in ['bull', 'bear', 'sideways']:
    mask = np.array([r == state for r in regimes])
    state_returns = returns[mask]
    regime_stats[state] = RegimeStats(
        mean_return=float(np.mean(state_returns)) if len(state_returns) > 0 else 0.0,
        std_return=float(np.std(state_returns, ddof=1)) if len(state_returns) > 1 else 0.01,
    )

for state, rs in regime_stats.items():
    print(f"{state:8s}: mu={rs.mean_return:.5f}  sigma={rs.std_return:.5f}")

## 3. Compute Trajectory

Generate a 7-day trajectory with 1,000 Monte Carlo paths.

In [ ]:
trajectory = compute_trajectory(
    current_price=current_price,
    days=HORIZON,
    P=P,
    regime_stats=regime_stats,
    initial_state=current_regime,
    n_samples=1000,
    nu=5,
)

traj_df = pd.DataFrame([
    {
        'day': pt.day,
        'expected': pt.expected_price,
        'lower': pt.lower_bound,
        'upper': pt.upper_bound,
        'p_up': pt.p_up,
        'regime': pt.regime,
        'cumulative_return': pt.cumulative_return,
    }
    for pt in trajectory
])
traj_df

## 4. Visualise Trajectory

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price trajectory with CI
ax = axes[0]
days = traj_df['day'].values
ax.fill_between(days, traj_df['lower'], traj_df['upper'], alpha=0.2, label='90% CI')
ax.plot(days, traj_df['expected'], linewidth=2, label='Expected')
ax.axhline(current_price, color='red', linestyle='--', label=f'Current: ${current_price:,.0f}')
ax.set_xlabel('Day')
ax.set_ylabel('Price')
ax.set_title(f'{TICKER} {HORIZON}-Day Trajectory')
ax.legend()
ax.grid(True, alpha=0.3)

# P(up) per day
ax = axes[1]
ax.bar(days, traj_df['p_up'] * 100, color='steelblue', alpha=0.7)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Neutral')
ax.set_xlabel('Day')
ax.set_ylabel('P(up) %')
ax.set_title(f'{TICKER}: Probability of Increase')
ax.set_ylim(0, 100)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Scenario Probabilities

Build a synthetic survival distribution from the trajectory and bucket probabilities.

In [ ]:
# Build a distribution from trajectory upper/lower bounds
distribution = []
for _, row in traj_df.iterrows():
    distribution.append({'price': row['lower'], 'probability': 0.9})
    distribution.append({'price': row['expected'], 'probability': 0.5})
    distribution.append({'price': row['upper'], 'probability': 0.1})

# Sort and deduplicate
distribution = sorted(distribution, key=lambda d: d['price'])

scenarios = compute_scenario_probabilities(distribution, current_price)

print(f"Expected price: ${scenarios['expected_price']:,.2f}")
print(f"Expected return: {scenarios['expected_return']*100:.2f}%")
print(f"P(up): {scenarios['p_up']:.3f}")
print()
for bucket in scenarios['buckets']:
    print(f"  {bucket['label']:12s}: {bucket['probability']*100:5.1f}%")

## 6. Structural Validation

Replicate the TypeScript integration-test assertions.

In [ ]:
# All values finite
for pt in trajectory:
    assert np.isfinite(pt.expected_price)
    assert np.isfinite(pt.lower_bound)
    assert np.isfinite(pt.upper_bound)
    assert np.isfinite(pt.p_up)
    assert pt.lower_bound < pt.expected_price < pt.upper_bound

# CI widths increase monotonically (with 1% price tolerance)
for i in range(1, len(trajectory)):
    prev_w = trajectory[i-1].upper_bound - trajectory[i-1].lower_bound
    curr_w = trajectory[i].upper_bound - trajectory[i].lower_bound
    assert curr_w >= prev_w - current_price * 0.01

# Day 1 expected price within 2% of current
day1 = trajectory[0]
pct_diff = abs(day1.expected_price - current_price) / current_price
assert pct_diff < 0.02, f'Day 1 expected price off by {pct_diff*100:.1f}%'

# P(up) in [0,1]; day 1 not extreme
for pt in trajectory:
    assert 0 <= pt.p_up <= 1
assert 0.15 <= day1.p_up <= 0.85

# Cumulative return magnitude reasonable
for pt in trajectory:
    ret = (pt.expected_price - current_price) / current_price
    assert abs(ret) < 0.20, f'Day {pt.day} return {ret*100:.1f}% too large'

print('All structural validations passed.')

## 7. Sensitivity: Sample Count

How does the trajectory change with more or fewer MC samples?

In [ ]:
sample_counts = [100, 500, 1000, 5000]
fig, ax = plt.subplots(figsize=(10, 5))

for n in sample_counts:
    traj = compute_trajectory(
        current_price=current_price,
        days=HORIZON,
        P=P,
        regime_stats=regime_stats,
        initial_state=current_regime,
        n_samples=n,
    )
    days_arr = [pt.day for pt in traj]
    expected = [pt.expected_price for pt in traj]
    ax.plot(days_arr, expected, label=f'n={n}', linewidth=2)

ax.axhline(current_price, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Day')
ax.set_ylabel('Expected Price')
ax.set_title(f'{TICKER}: Trajectory Sensitivity to Sample Count')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()